In [45]:
from pathlib import Path

import geopandas as gpd
import pandas as pd

# Insert the date (YYYY-MM-DD) of the VIIRS file you want to process
DATE = "2026-05-02"
PARENT_PATH = Path.cwd().parent

## Load and pre-process data

### Fire data csv
---
I will first load the fire data csv and inspect its properties

In [46]:
fire_data = pd.read_csv(
    PARENT_PATH.joinpath("data/raw", f"VIIRS_SNNP_NRT_world_14days_{DATE}.csv")
)

fire_data.info()
fire_data.head()

<class 'pandas.DataFrame'>
RangeIndex: 389107 entries, 0 to 389106
Data columns (total 14 columns):
 #   Column      Non-Null Count   Dtype  
---  ------      --------------   -----  
 0   latitude    389107 non-null  float64
 1   longitude   389107 non-null  float64
 2   bright_ti4  389107 non-null  float64
 3   scan        389107 non-null  float64
 4   track       389107 non-null  float64
 5   acq_date    389107 non-null  str    
 6   acq_time    389107 non-null  int64  
 7   satellite   389107 non-null  str    
 8   instrument  389107 non-null  str    
 9   confidence  389107 non-null  str    
 10  version     389107 non-null  str    
 11  bright_ti5  389107 non-null  float64
 12  frp         389107 non-null  float64
 13  daynight    389107 non-null  str    
dtypes: float64(7), int64(1), str(6)
memory usage: 41.6 MB


,latitude,longitude,bright_ti4,scan,track,acq_date,acq_time,satellite,instrument,confidence,version,bright_ti5,frp,daynight
0,32.33215,44.09279,306.72,0.71,0.75,2026-05-01,1,N,VIIRS,n,2.0NRT,288.92,2.80,N
1,32.88856,35.09304,296.63,0.44,0.46,2026-05-01,1,N,VIIRS,n,2.0NRT,281.80,1.02,N
2,33.15357,44.78751,302.25,0.73,0.76,2026-05-01,1,N,VIIRS,n,2.0NRT,287.02,2.17,N
3,33.15594,44.77987,316.28,0.73,0.76,2026-05-01,1,N,VIIRS,n,2.0NRT,288.05,2.17,N
4,33.15751,44.78342,342.93,0.73,0.76,2026-05-01,1,N,VIIRS,n,2.0NRT,289.27,6.42,N


I will now drop VIIRS records with a low confidence and unnnecessary columns

In [47]:
fire_data = fire_data[fire_data["confidence"] != "l"]
fire_data.drop(columns=["satellite", "instrument", "version", "acq_time"], inplace=True)

fire_data.info()

<class 'pandas.DataFrame'>
Index: 337837 entries, 0 to 389106
Data columns (total 10 columns):
 #   Column      Non-Null Count   Dtype  
---  ------      --------------   -----  
 0   latitude    337837 non-null  float64
 1   longitude   337837 non-null  float64
 2   bright_ti4  337837 non-null  float64
 3   scan        337837 non-null  float64
 4   track       337837 non-null  float64
 5   acq_date    337837 non-null  str    
 6   confidence  337837 non-null  str    
 7   bright_ti5  337837 non-null  float64
 8   frp         337837 non-null  float64
 9   daynight    337837 non-null  str    
dtypes: float64(7), str(3)
memory usage: 28.4 MB


I finally compute the true area of the pixel in which the wildfire is detected. The pixel size at nadir is 375 meters, and as NASA documentation for [VIIRS attributes](https://www.earthdata.nasa.gov/data/tools/firms/active-fire-data-attributes-modis-viirs#toc-attribute-fields-for-nrt-viirs-375m-active-fire-datadocumentation) points out, scan and track reflect the actual pixel area. I also keep only the columns I need for further analysis.

In [48]:
fire_data["fire_area"] = fire_data["scan"] * fire_data["track"]
fire_data = fire_data[["acq_date", "fire_area", "latitude", "longitude"]]
fire_data.info()

<class 'pandas.DataFrame'>
Index: 337837 entries, 0 to 389106
Data columns (total 4 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   acq_date   337837 non-null  str    
 1   fire_area  337837 non-null  float64
 2   latitude   337837 non-null  float64
 3   longitude  337837 non-null  float64
dtypes: float64(3), str(1)
memory usage: 12.9 MB


 I now create a GeoDataFrame from the csv

In [49]:
fire_data = gpd.GeoDataFrame(
    fire_data,
    geometry=gpd.points_from_xy(fire_data.longitude, fire_data.latitude),
    crs="EPSG:4326",
)

fire_data = fire_data.drop(columns=["latitude", "longitude"])
fire_data.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
Index: 337837 entries, 0 to 389106
Data columns (total 3 columns):
 #   Column     Non-Null Count   Dtype   
---  ------     --------------   -----   
 0   acq_date   337837 non-null  str     
 1   fire_area  337837 non-null  float64 
 2   geometry   337837 non-null  geometry
dtypes: float64(1), geometry(1), str(1)
memory usage: 10.3 MB


### Countries area csv
---
I first load the dataset with the countries' areas and inspect it.

In [50]:
countries_area = pd.read_csv(PARENT_PATH.joinpath("data/raw", "surface_area.csv"))

countries_area.info()
countries_area.head()

<class 'pandas.DataFrame'>
RangeIndex: 16410 entries, 0 to 16409
Data columns (total 41 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   STRUCTURE               16410 non-null  str    
 1   STRUCTURE_ID            16410 non-null  str    
 2   ACTION                  16410 non-null  str    
 3   FREQ                    16410 non-null  str    
 4   REF_AREA                16410 non-null  str    
 5   INDICATOR               16410 non-null  str    
 6   SEX                     16410 non-null  str    
 7   AGE                     16410 non-null  str    
 8   URBANISATION            16410 non-null  str    
 9   UNIT_MEASURE            16410 non-null  str    
 10  COMP_BREAKDOWN_1        16410 non-null  str    
 11  COMP_BREAKDOWN_2        16410 non-null  str    
 12  COMP_BREAKDOWN_3        16410 non-null  str    
 13  TIME_PERIOD             16410 non-null  int64  
 14  OBS_VALUE               16410 non-null  float64
 

,STRUCTURE,STRUCTURE_ID,ACTION,FREQ,REF_AREA,INDICATOR,SEX,AGE,URBANISATION,UNIT_MEASURE,...,COMP_BREAKDOWN_2_LABEL,COMP_BREAKDOWN_3_LABEL,AGG_METHOD_LABEL,UNIT_TYPE_LABEL,DECIMALS_LABEL,DATABASE_ID_LABEL,TIME_FORMAT_LABEL,UNIT_MULT_LABEL,OBS_STATUS_LABEL,OBS_CONF_LABEL
0,datastructure,WB.DATA360:DS_DATA360(1.3),I,A,URY,WB_WDI_AG_SRF_TOTL_K2,_T,_T,_T,KM2,...,Not Applicable,Not Applicable,Not Applicable,Number (real number),Two,World Development Indicators (WDI),Annual,Units,Normal value,Public
1,datastructure,WB.DATA360:DS_DATA360(1.3),I,A,UZB,WB_WDI_AG_SRF_TOTL_K2,_T,_T,_T,KM2,...,Not Applicable,Not Applicable,Not Applicable,Number (real number),Two,World Development Indicators (WDI),Annual,Units,Normal value,Public
2,datastructure,WB.DATA360:DS_DATA360(1.3),I,A,VUT,WB_WDI_AG_SRF_TOTL_K2,_T,_T,_T,KM2,...,Not Applicable,Not Applicable,Not Applicable,Number (real number),Two,World Development Indicators (WDI),Annual,Units,Normal value,Public
3,datastructure,WB.DATA360:DS_DATA360(1.3),I,A,VEN,WB_WDI_AG_SRF_TOTL_K2,_T,_T,_T,KM2,...,Not Applicable,Not Applicable,Not Applicable,Number (real number),Two,World Development Indicators (WDI),Annual,Units,Normal value,Public
4,datastructure,WB.DATA360:DS_DATA360(1.3),I,A,VNM,WB_WDI_AG_SRF_TOTL_K2,_T,_T,_T,KM2,...,Not Applicable,Not Applicable,Not Applicable,Number (real number),Two,World Development Indicators (WDI),Annual,Units,Normal value,Public


I now filter out the huge amount of unnecessary attributes, cast strings into correct types and rename columns to more descriptive names.
I also keep only area data from the last surveyed year (2023).

In [51]:
countries_area = countries_area[countries_area["TIME_PERIOD"] == 2023]

In [52]:
countries_area = countries_area[["REF_AREA", "OBS_VALUE"]]
countries_area["OBS_VALUE"] = countries_area["OBS_VALUE"].astype(float)
countries_area = countries_area.rename(
    columns={"OBS_VALUE": "country_area", "REF_AREA": "iso_code"}
)

countries_area.info()

<class 'pandas.DataFrame'>
Index: 215 entries, 1744 to 9791
Data columns (total 2 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   iso_code      215 non-null    str    
 1   country_area  215 non-null    float64
dtypes: float64(1), str(1)
memory usage: 5.0 KB


### Countries' boundaries GeoJson
---
I load the GeoJson into a dataframe and check the validity of the countries' geometries. If necessary, I try to repair them.

In [53]:
countries_boundaries = gpd.read_file(
    PARENT_PATH.joinpath("data/raw", "geoboundaries_world.geojson")
).to_crs("EPSG:4326")

# check for invalid geometries and repair them if necessary
if not countries_boundaries.is_valid.all():
    countries_boundaries.make_valid()
    print("Repairing invalid geometries")
if not countries_boundaries.is_valid.all():
    raise Exception("Some geometries are invalid and not reparable")

countries_boundaries.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 218 entries, 0 to 217
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype   
---  ------      --------------  -----   
 0   shapeGroup  218 non-null    str     
 1   shapeType   218 non-null    str     
 2   shapeName   218 non-null    str     
 3   geometry    218 non-null    geometry
dtypes: geometry(1), str(3)
memory usage: 6.9 KB


I rename the columns to more descriptive names and remove unnecessary columns

In [54]:
countries_boundaries = countries_boundaries.rename(
    columns={"shapeGroup": "iso_code", "shapeName": "name"}
)
countries_boundaries = countries_boundaries.drop(columns=["shapeType"])

countries_boundaries.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 218 entries, 0 to 217
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype   
---  ------    --------------  -----   
 0   iso_code  218 non-null    str     
 1   name      218 non-null    str     
 2   geometry  218 non-null    geometry
dtypes: geometry(1), str(2)
memory usage: 5.2 KB


## Processing and join of the datasets
### Join countries' boundaries and area dataset
---

I now join countries' boundaries with the area dataset

In [55]:
countries_boundaries_area = pd.merge(
    countries_boundaries,
    countries_area,
    left_on="iso_code",
    right_on="iso_code",
    how="left",
)

countries_boundaries_area.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 218 entries, 0 to 217
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype   
---  ------        --------------  -----   
 0   iso_code      218 non-null    str     
 1   name          218 non-null    str     
 2   geometry      218 non-null    geometry
 3   country_area  194 non-null    float64 
dtypes: float64(1), geometry(1), str(2)
memory usage: 6.9 KB


Some country area values are `NA` values. Why is so?

In [56]:
selected = countries_boundaries_area.loc[
    countries_boundaries_area["country_area"].isna()
]
display(selected)

,iso_code,name,geometry,country_area
5,ATA,Antarctica,"MULTIPOLYGON (((-60.06171 -79.6813, -60.05473 ...",NaN
94,XKX,Kosovo,"POLYGON ((20.59429 41.87733, 20.5955 41.8765, ...",NaN
168,TWN,Taiwan,"MULTIPOLYGON (((116.71997 20.70818, 116.71902 ...",NaN
179,VAT,Vatican City,"POLYGON ((12.4538 41.90682, 12.45308 41.90668,...",NaN
198,111,Abyei,"POLYGON ((29 9.67356, 29 10.16667, 27.83333 10...",NaN
199,112,Aksai Chin,"MULTIPOLYGON (((78.69839 34.09307, 78.69837 34...",NaN
200,113,CH-IN,"MULTIPOLYGON (((79.70073 30.97073, 79.70088 30...",NaN
201,114,Demchok,"POLYGON ((79.15197 33.18187, 79.15271 33.18106...",NaN
202,115,Dragonja,"MULTIPOLYGON (((13.67648 45.44426, 13.67648 45...",NaN
203,116,Dramana-Shakatoe,"POLYGON ((89.12804 27.61484, 89.12799 27.61484...",NaN


As we can observe the countries without an area are those not officially recognized by the UN. Since the area dataset has been created with data fro the FAO, those countries were missing in the table. Since I need the area to later compute the percentage of the land surface affected by wildfires, I will just drop them. With insight, I could have joined the tables with a right join to drop the non-matching left tuples.

In [57]:
countries_boundaries_area = countries_boundaries_area.dropna()
countries_boundaries_area.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
Index: 194 entries, 0 to 197
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype   
---  ------        --------------  -----   
 0   iso_code      194 non-null    str     
 1   name          194 non-null    str     
 2   geometry      194 non-null    geometry
 3   country_area  194 non-null    float64 
dtypes: float64(1), geometry(1), str(2)
memory usage: 7.6 KB


### Compute the total wildfire-affected area for each country
---
I now spatially join the countries geometries with the wildfire data. I decided to use the default spatial predicate `intersects`: the points can only intersect one country's geometries at a time. I join them in a left join to drop wildfire tuples that don't intersect with a country geometry.
As explained by inline comments, after the join I used pandas `groupby()` method to aggregate the total wildfire area per country, since using geopandas `dissolve()` was timing out and crashing the kernel. I didn't investigated further and used pandas' method as workaround. Because of this, I had to rejoin the aggregated fire data with the countries' geometries.

In [58]:
fire_data_countries = gpd.sjoin(
    countries_boundaries_area,
    fire_data,
    how="left",
)

# dissolve() times out and crashes the kernel, worked around by using df.groupby() and joining
# back later with the countries geometries on the iso code key

fire_data_by_countries = pd.DataFrame(
    fire_data_countries[["iso_code", "name", "country_area", "fire_area"]]
    .groupby(by=["iso_code", "name", "country_area"])["fire_area"]
    .sum()
    .reset_index()
)

fire_data_by_countries = gpd.GeoDataFrame(
    fire_data_by_countries.merge(
        countries_boundaries, on=["iso_code", "name"], how="left"
    )
)

fire_data_by_countries.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 194 entries, 0 to 193
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype   
---  ------        --------------  -----   
 0   iso_code      194 non-null    str     
 1   name          194 non-null    str     
 2   country_area  194 non-null    float64 
 3   fire_area     194 non-null    float64 
 4   geometry      194 non-null    geometry
dtypes: float64(2), geometry(1), str(2)
memory usage: 7.7 KB


I now finally compute the percentage of wildfire-affected area

In [59]:
fire_data_by_countries["area_perc"] = (
    fire_data_by_countries["fire_area"] / fire_data_by_countries["country_area"]
) * 100

fire_data_by_countries.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 194 entries, 0 to 193
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype   
---  ------        --------------  -----   
 0   iso_code      194 non-null    str     
 1   name          194 non-null    str     
 2   country_area  194 non-null    float64 
 3   fire_area     194 non-null    float64 
 4   geometry      194 non-null    geometry
 5   area_perc     194 non-null    float64 
dtypes: float64(3), geometry(1), str(2)
memory usage: 9.2 KB


## Simplify polygons
Using the non-simplified polygons the resulting html map created with folium were huge and slow. After further debugging with a LLM and Stackoverflow threads, I understood that the polygons were too complex. In retrospect, the complex polygons could also explain the above mentioned kernel crash when trying to dissolve the countries geometries. The kernel ran out of working memory and crashed.

In [60]:
# TODO optimize memory: write to disk as soon as simplify is done and use del keyword to manually decrease reference count and make garbage collector deallocate the objects

fire_data_by_countries_simplified = fire_data_by_countries.copy()
fire_data_by_countries_simplified["geometry"] = (
    fire_data_by_countries.geometry.simplify_coverage(tolerance=0.05)
)
countries_boundaries_simplified = countries_boundaries.copy()
countries_boundaries_simplified["geometry"] = (
    countries_boundaries.geometry.simplify_coverage(tolerance=0.05)
)

Unlike `simplify()`, `simplify_coverage()` assumes that the GeoSeries forms a polygonal coverage. This allows polygons borders to remain consistent by avoiding overlaps and holes.

## Save processed data

In [61]:
fire_data_by_countries_simplified.to_file(
    PARENT_PATH.joinpath("data/processed", "fire_data_by_countries.gpkg")
)
countries_boundaries_simplified.to_file(
    PARENT_PATH.joinpath("data/processed", "countries_boundaries_simplified.gpkg")
)